# LeetCode 3897 — Maximum Value of Concatenated Binary Segments bằng Python

- Link: https://leetcode.com/problems/maximum-value-of-concatenated-binary-segments/
- Độ khó: **Hard**
- Chủ đề: Tham lam (greedy) bằng sắp xếp, so sánh "AB vs BA", lập luận đổi chỗ (exchange argument), lũy thừa modulo, quy tắc Horner

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxValue(self, nums1: list[int], nums0: list[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: tổng độ dài xâu **cố định**, nên "giá trị nhị phân lớn nhất" ⇔ "xâu **lớn nhất theo thứ tự từ điển**".
2. Áp dụng khuôn **sắp xếp theo so sánh `A+B` với `B+A`** (giống *Largest Number*, LC 179) bằng `functools.cmp_to_key`.
3. Rút gọn so sánh đó thành **khóa sắp xếp đơn giản** cho đoạn dạng `1^x 0^y`: toàn 1 trước, rồi `x` giảm dần / `y` tăng dần, toàn 0 cuối.
4. Chứng minh đúng bằng **lập luận đổi chỗ** (exchange argument).
5. Tính giá trị xâu dài `2·10^5` bit theo modulo bằng lũy thừa 2 hoặc **Horner** mà không dựng xâu.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n` đoạn; đoạn `i` là xâu nhị phân gồm `nums1[i]` chữ `1` **rồi** `nums0[i]` chữ `0`. Ta được **sắp xếp lại thứ tự** các đoạn tùy ý, rồi nối lại thành một xâu nhị phân. Tìm giá trị số nguyên **lớn nhất** của xâu đó, theo modulo `10^9 + 7`.

**Ví dụ 1:** `nums1 = [1,2], nums0 = [1,0]` → `14`. Đoạn `"10"` và `"11"`; nối `"11" + "10" = "1110" = 14`.

**Ví dụ 2:** `nums1 = [3,1], nums0 = [0,3]` → `120`. `"111" + "1000" = "1111000" = 120`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `0 ≤ nums1[i], nums0[i] ≤ 10^4`, `nums1[i] + nums0[i] > 0`, tổng mọi phần tử `≤ 2·10^5`.

In [1]:
from functools import cmp_to_key
from itertools import permutations
import random, time

MOD = 10**9 + 7
examples = [([1, 2], [1, 0], 14), ([3, 1], [0, 3], 120)]
def seg(x, y):
    return '1' * x + '0' * y
for a, b, exp in examples:
    print([seg(x, y) for x, y in zip(a, b)], "-> kỳ vọng", exp)

['10', '11'] -> kỳ vọng 14
['111', '1000'] -> kỳ vọng 120


## 3. Kiến thức nền

### 3.1. Xâu nhị phân cùng độ dài: so sánh giá trị = so sánh từ điển

Hai xâu nhị phân **cùng độ dài** (cho phép số 0 đầu): xâu nào lớn hơn theo từ điển thì giá trị lớn hơn. Vì mọi cách sắp xếp cho cùng tổng độ dài, ta chỉ cần xâu lớn nhất theo từ điển.

In [2]:
for s, t in [("0110", "0101"), ("1000", "0111"), ("1110", "1101")]:
    print(s, t, "| so từ điển:", s > t, "| so giá trị:", int(s, 2) > int(t, 2))

0110 0101 | so từ điển: True | so giá trị: True
1000 0111 | so từ điển: True | so giá trị: True
1110 1101 | so từ điển: True | so giá trị: True


### 3.2. `functools.cmp_to_key` — sắp xếp theo hàm so sánh

`cmp(a, b)` trả số âm nếu `a` nên đứng trước `b`, dương nếu sau, `0` nếu như nhau. `sorted(lst, key=cmp_to_key(cmp))`.

In [3]:
def cmp_concat(a, b):
    if a + b > b + a: return -1
    if a + b < b + a: return 1
    return 0

print(sorted(["10", "11", "1000", "0"], key=cmp_to_key(cmp_concat)))

['11', '10', '1000', '0']


### 3.3. Lũy thừa 2 theo modulo và quy tắc Horner

- Bit `1` ở vị trí `k` (tính từ phải, bắt đầu 0) đóng góp `2^k`. Tiền tính `pw[k] = 2^k mod M`.
- **Horner:** đọc xâu từ trái sang phải, `val = val · 2^{len(đoạn)} + giá_trị(đoạn)`. Đoạn `1^x 0^y` có giá trị `(2^x - 1) · 2^y`.

In [4]:
s = "1111000"
pw = [1] * len(s)
for k in range(1, len(s)): pw[k] = pw[k - 1] * 2 % MOD
v1 = sum(pw[len(s) - 1 - i] for i, c in enumerate(s) if c == '1') % MOD
v2 = 0
for x, y in [(3, 0), (1, 3)]:
    v2 = (v2 * pow(2, x + y, MOD) + (pow(2, x, MOD) - 1) * pow(2, y, MOD)) % MOD
print(int(s, 2), v1, v2)

120 120 120


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn số lớn nhất ⇒ muốn **càng nhiều chữ 1 càng sớm**.

- Đoạn **toàn 1** (`y = 0`) không chèn chữ 0 nào ⇒ đặt **đầu tiên**.
- Đoạn **toàn 0** (`x = 0`) chỉ làm chậm chữ 1 ⇒ đặt **cuối cùng**.
- Đoạn **hỗn hợp** `1^x 0^y`: đoạn có nhiều 1 hơn nên đặt trước (dãy 1 ở đầu dài hơn). Nếu cùng số 1, đoạn ít 0 hơn đặt trước (chữ 1 của đoạn sau đến sớm hơn).

Tính tay: đoạn `"110"`, `"1100"`, `"10"`, `"111"`, `"00"`:

- Toàn 1: `"111"`. Hỗn hợp theo `x` giảm, `y` tăng: `"110"`, `"1100"`, `"10"`. Toàn 0: `"00"`.
- Kết quả `"111" "110" "1100" "10" "00"` = `"111110110010" + "00"`.

In [5]:
segs = ["110", "1100", "10", "111", "00"]
best = max(("".join(p) for p in permutations(segs)), key=lambda s: int(s, 2))
print("tốt nhất (vét cạn):", best)
print("theo trực giác   :", "111" + "110" + "1100" + "10" + "00")

tốt nhất (vét cạn): 11111011001000
theo trực giác   : 11111011001000


## 5. Cách 1 — Brute Force: thử mọi hoán vị

`n!` hoán vị, mỗi hoán vị dựng xâu và chuyển sang số. Chỉ dùng cho `n ≤ 8`.

In [6]:
def brute(nums1, nums0):
    segs = [seg(x, y) for x, y in zip(nums1, nums0)]
    best = max(int("".join(p), 2) for p in permutations(segs))
    return best % MOD

for a, b, exp in examples:
    print(brute(a, b), "kỳ vọng", exp)

14 kỳ vọng 14
120 kỳ vọng 120


**Nút thắt:** `n!` hoán vị. Ta cần một **thứ tự tham lam** — sắp xếp theo một quy tắc so sánh hai đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — So sánh cặp (exchange argument).** Trong một thứ tự tối ưu, với hai đoạn **kề nhau** `A, B`, phải có `A + B ≥ B + A` (nếu không, đổi chỗ hai đoạn làm xâu lớn hơn mà các phần khác giữ nguyên). Nếu quan hệ "`A` trước `B` ⇔ `A+B ≥ B+A`" là một **thứ tự toàn phần** (bắc cầu), thì sắp xếp theo nó cho đáp án tối ưu — đây là khuôn của LC 179.

**Bước 2 — Rút gọn cho đoạn `1^x 0^y`.** Với `A = 1^a 0^b`, `B = 1^c 0^d`:

| trường hợp | kết luận |
|---|---|
| `b = 0` (A toàn 1), `d > 0`, `c > 0` | `A+B = 1^{a+c}0^d` > `B+A` ⇒ **A trước** |
| `a > c`, `b, d > 0` | `A+B` có `a` chữ 1 đầu, `B+A` chỉ `c` ⇒ **A trước** |
| `a = c`, `b < d` | sau `1^a 0^b`, `A+B` gặp chữ `1` trong khi `B+A` vẫn là `0` ⇒ **A trước** |
| `a = 0` (A toàn 0) | A bắt đầu bằng 0, mọi đoạn có `1` nên đứng trước ⇒ **A cuối** |

⇒ **Khóa sắp xếp:** `(nhóm, …)` với nhóm 0 = toàn 1, nhóm 1 = hỗn hợp sắp `(-x, y)`, nhóm 2 = toàn 0.

**Bước 3 — Tính giá trị không dựng xâu:** cộng `2^{vị trí}` cho mỗi chữ 1, hoặc Horner theo từng đoạn.

## 7. Các cách tiếp theo

- **Cách 2:** sắp xếp bằng `cmp_to_key` so sánh `A+B` với `B+A` (dựng xâu thật) — dễ nghĩ, chậm hơn.
- **Cách 3:** sắp xếp bằng **khóa đơn giản** + cộng lũy thừa 2 cho từng bit 1 (lời giải nộp chính).
- **Cách 4:** cùng khóa, nhưng tính giá trị bằng **Horner theo đoạn** với `pow` — `O(n log)` thay vì `O(L)`.

### 7.1. Cách 2 — So sánh `A+B` và `B+A` bằng xâu thật

In [7]:
def concat_compare(nums1, nums0):
    segs = [seg(x, y) for x, y in zip(nums1, nums0)]
    segs.sort(key=cmp_to_key(cmp_concat))
    s = "".join(segs)
    return int(s, 2) % MOD if s else 0

for a, b, exp in examples:
    print(concat_compare(a, b), "kỳ vọng", exp)

14 kỳ vọng 14
120 kỳ vọng 120


**Độ phức tạp:** mỗi so sánh tốn `O(|A| + |B|)`, sắp xếp `O(n log n)` lần so sánh ⇒ có thể tới `O(L log n)` (với `L` tổng độ dài) nhưng tệ hơn khi vài đoạn rất dài bị so sánh nhiều lần. `int(s, 2)` với `L = 2·10^5` bit vẫn nhanh trong Python.

In [8]:
class SolutionCompare:
    def maxValue(self, nums1: list[int], nums0: list[int]) -> int:
        return concat_compare(nums1, nums0)

print([SolutionCompare().maxValue(a, b) for a, b, _ in examples])

[14, 120]


### 7.2. Cách 3 — Khóa sắp xếp + cộng lũy thừa 2 (lời giải chính)

**Giải thích từng đoạn:**

1. `key(x, y)`: `(0, -x, 0)` nếu `y = 0`; `(1, -x, y)` nếu `x > 0, y > 0`; `(2, 0, 0)` nếu `x = 0`.
2. `L = Σ (x + y)`; tiền tính `pw[k] = 2^k mod M` với `k < L`.
3. Duyệt các đoạn theo thứ tự; con trỏ `pos = L - 1` (vị trí bit cao nhất). Mỗi chữ `1`: cộng `pw[pos]`, giảm `pos`. Mỗi đoạn xong phần `1` thì trừ `pos` thêm `y`.

In [9]:
def seg_key(p):
    x, y = p
    if y == 0:
        return (0, -x, 0)
    if x > 0:
        return (1, -x, y)
    return (2, 0, 0)

def key_sort_bits(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=seg_key)
    L = sum(nums1) + sum(nums0)
    pw = [1] * max(L, 1)
    for k in range(1, L):
        pw[k] = pw[k - 1] * 2 % MOD
    pos = L - 1
    ans = 0
    for x, y in pairs:
        for _ in range(x):
            ans += pw[pos]
            pos -= 1
        pos -= y
    return ans % MOD

for a, b, exp in examples:
    print(key_sort_bits(a, b), "kỳ vọng", exp)

14 kỳ vọng 14
120 kỳ vọng 120


**Độ phức tạp Cách 3:** sắp xếp `O(n log n)` + duyệt bit `O(L)` ⇒ `O(n log n + L)`, bộ nhớ `O(n + L)`.

### 7.3. Cách 4 — Khóa sắp xếp + Horner theo đoạn

Đọc các đoạn theo thứ tự: `val = val · 2^{x+y} + (2^x - 1) · 2^y` (mod `M`). Không cần mảng `pw`.

In [10]:
def key_sort_horner(nums1, nums0):
    val = 0
    for x, y in sorted(zip(nums1, nums0), key=seg_key):
        val = (val * pow(2, x + y, MOD) + (pow(2, x, MOD) - 1) * pow(2, y, MOD)) % MOD
    return val

for a, b, exp in examples:
    print(key_sort_horner(a, b), "kỳ vọng", exp)

14 kỳ vọng 14
120 kỳ vọng 120


In [11]:
class SolutionHorner:
    def maxValue(self, nums1: list[int], nums0: list[int]) -> int:
        return key_sort_horner(nums1, nums0)

print([SolutionHorner().maxValue(a, b) for a, b, _ in examples])

[14, 120]


**Độ phức tạp Cách 4:** `O(n log n + n log L)` (mỗi `pow` là `O(log L)`), bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums1 = [2, 0, 1, 3, 2]`, `nums0 = [1, 2, 1, 0, 2]` (ví dụ tự dựng).

| đoạn | (x, y) | nhóm | khóa |
|---|---|---|---|
| "110" | (2, 1) | hỗn hợp | (1, −2, 1) |
| "00" | (0, 2) | toàn 0 | (2, 0, 0) |
| "10" | (1, 1) | hỗn hợp | (1, −1, 1) |
| "111" | (3, 0) | toàn 1 | (0, −3, 0) |
| "1100" | (2, 2) | hỗn hợp | (1, −2, 2) |

Sau sắp xếp: `"111"`, `"110"`, `"1100"`, `"10"`, `"00"` ⇒ xâu `11111011001000`.

In [12]:
def trace(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=seg_key)
    print("thứ tự:", [seg(x, y) for x, y in pairs])
    L = sum(nums1) + sum(nums0)
    pos, ans = L - 1, 0
    for x, y in pairs:
        before = ans
        for _ in range(x):
            ans += 2 ** pos
            pos -= 1
        pos -= y
        print(f"đoạn {seg(x, y):<6} cộng {ans - before:>6} (bit 1 ở các vị trí cao), tổng = {ans} = {bin(ans)[2:].zfill(L)}")
    print("kết quả:", ans, "| vét cạn:", brute(nums1, nums0))

trace([2, 0, 1, 3, 2], [1, 2, 1, 0, 2])

thứ tự: ['111', '110', '1100', '10', '00']
đoạn 111    cộng  14336 (bit 1 ở các vị trí cao), tổng = 14336 = 11100000000000
đoạn 110    cộng   1536 (bit 1 ở các vị trí cao), tổng = 15872 = 11111000000000
đoạn 1100   cộng    192 (bit 1 ở các vị trí cao), tổng = 16064 = 11111011000000
đoạn 10     cộng      8 (bit 1 ở các vị trí cao), tổng = 16072 = 11111011001000
đoạn 00     cộng      0 (bit 1 ở các vị trí cao), tổng = 16072 = 11111011001000
kết quả: 16072 | vét cạn: 16072


In [13]:
# Trace Horner trên ví dụ 2
val = 0
for x, y in sorted(zip([3, 1], [0, 3]), key=seg_key):
    old = val
    val = val * 2 ** (x + y) + (2 ** x - 1) * 2 ** y
    print(f"đoạn {seg(x, y)}: {old} * 2^{x + y} + {(2 ** x - 1) * 2 ** y} = {val} ({bin(val)[2:]})")

đoạn 111: 0 * 2^3 + 7 = 7 (111)
đoạn 1000: 7 * 2^4 + 8 = 120 (1111000)


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxValue(self, nums1: list[int], nums0: list[int]) -> int:
        MOD = 10**9 + 7

        def key(p):
            x, y = p
            if y == 0:
                return (0, -x, 0)      # toàn 1: đầu tiên
            if x > 0:
                return (1, -x, y)      # hỗn hợp: nhiều 1 trước, ít 0 trước
            return (2, 0, 0)           # toàn 0: cuối cùng

        pairs = sorted(zip(nums1, nums0), key=key)
        L = sum(nums1) + sum(nums0)
        pw = [1] * L
        for k in range(1, L):
            pw[k] = pw[k - 1] * 2 % MOD
        pos = L - 1
        ans = 0
        for x, y in pairs:
            for _ in range(x):
                ans += pw[pos]
                pos -= 1
            pos -= y
        return ans % MOD

sol = Solution()
for a, b, exp in examples:
    got = sol.maxValue(a, b)
    print(got, "✔" if got == exp else "✘")

14 ✔
120 ✔


## 10. Vì sao thuật toán đúng

1. **Cùng độ dài ⇒ so từ điển:** mọi thứ tự cho xâu độ dài `L`; giá trị lớn nhất ⇔ xâu lớn nhất theo từ điển.
2. **Exchange argument:** xét thứ tự tối ưu bất kỳ; nếu có hai đoạn kề nhau `A, B` mà `B` có khóa nhỏ hơn `A` (theo bảng ở mục 6 ⇒ `B + A > A + B`), đổi chỗ chúng: phần trước và sau không đổi, phần giữa tăng ⇒ xâu tăng ngặt, mâu thuẫn. Nếu khóa bằng nhau thì hai đoạn **giống hệt** (cùng `x, y`) — hoặc cùng nhóm toàn 0 (đổi chỗ không đổi xâu) — nên đổi chỗ không thay đổi gì.
3. **Bắc cầu:** khóa là tuple, so sánh tuple là thứ tự toàn phần ⇒ sắp xếp hợp lệ. Vì thứ tự sau sắp xếp không có cặp kề nào "đổi chỗ được lợi", và mọi thứ tự tối ưu có thể đưa về thứ tự đã sắp bằng các phép đổi chỗ không làm giảm, nên thứ tự đã sắp là tối ưu.
4. **Tính giá trị:** chữ 1 ở vị trí `pos` (tính từ phải) đóng góp `2^{pos}`; duyệt trái → phải với `pos` giảm dần đúng bằng vị trí bit.

Lưu ý tinh tế: đoạn toàn 1 với nhau đứng thứ tự nào cũng như nhau (đều là chuỗi 1), khóa `-x` chỉ để cố định thứ tự.

In [15]:
# Kiểm chứng bằng máy: với mọi cặp đoạn nhỏ, thứ tự theo khóa trùng với so sánh A+B vs B+A
bad = []
pieces = [(x, y) for x in range(0, 6) for y in range(0, 6) if x + y > 0]
for A in pieces:
    for B in pieces:
        sa, sb = seg(*A), seg(*B)
        key_says = seg_key(A) <= seg_key(B)          # A đứng trước (hoặc ngang) B theo khóa
        concat_says = sa + sb >= sb + sa             # A trước B không làm xâu nhỏ đi
        if key_says and not concat_says:
            bad.append((sa, sb))
print("số cặp mà khóa xếp A trước nhưng A+B < B+A:", len(bad), "trên", len(pieces) ** 2, "cặp")

số cặp mà khóa xếp A trước nhưng A+B < B+A: 0 trên 1225 cặp


**Vì sao cần tính bắc cầu?** Nếu hàm so sánh không bắc cầu (ví dụ `A < B`, `B < C` nhưng `C < A`), thuật toán sắp xếp có thể trả về thứ tự tùy thuộc vào thứ tự đầu vào, và lập luận đổi chỗ cặp kề nhau không còn kéo theo tối ưu toàn cục. Với khóa tuple, tính bắc cầu là hiển nhiên. Với `A+B` vs `B+A` trên xâu tùy ý, tính bắc cầu cũng đúng (một kết quả kinh điển của bài *Largest Number*), nhưng chứng minh dài hơn — đó là lý do nên rút gọn về khóa khi có thể.

Ngoài ra, so sánh bằng khóa còn giúp tránh **dựng xâu** dài `|A| + |B|` ở mỗi lần so sánh: với vài đoạn dài `10^4` ký tự, Cách 2 tốn nhiều bộ nhớ tạm hơn hẳn.

## 11. Phân tích Time / Space Complexity

Đặt `n` = số đoạn, `L = Σ (x + y) ≤ 2·10^5`.

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị | `O(n! · L)` | `O(L)` |
| 2. So sánh `A+B` / `B+A` | `O(n log n · độ dài đoạn)` + `O(L)` | `O(L)` |
| 3. Khóa + lũy thừa 2 | `O(n log n + L)` | `O(n + L)` |
| 4. Khóa + Horner | `O(n log n + n log L)` | `O(n)` |

## 12. Bộ test

In [16]:
def run_all(a, b, with_brute=True):
    res = [concat_compare(a, b), key_sort_bits(a, b), key_sort_horner(a, b), Solution().maxValue(a, b)]
    if with_brute:
        res.append(brute(a, b))
    return res

for a, b, exp in examples:
    assert all(v == exp for v in run_all(a, b))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [17]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([0], [5]),                 # một đoạn toàn 0 -> 0
    ([4], [0]),                 # một đoạn toàn 1 -> 15
    ([0, 0], [1, 2]),           # chỉ có đoạn toàn 0
    ([1, 1, 1], [2, 1, 3]),     # cùng x, khác y -> y tăng dần
    ([2, 3, 0, 1], [0, 0, 3, 0]),
    ([10**4], [10**4]),         # đoạn dài
]
for a, b in cases:
    r = run_all(a, b, with_brute=len(a) <= 7)
    assert len(set(r)) == 1, (a, b, r)
    print(a, b, "->", r[0])
assert Solution().maxValue([4], [0]) == 15 and Solution().maxValue([0], [5]) == 0

[0] [5] -> 0
[4] [0] -> 15
[0, 0] [1, 2] -> 0
[1, 1, 1] [2, 1, 3] -> 328
[2, 3, 0, 1] [0, 0, 3, 0] -> 504
[10000] [10000] -> 708817082


In [18]:
# 12.3 phản ví dụ: sắp xếp theo "tỉ lệ số 1" hoặc theo độ dài giảm dần đều sai
def wrong_ratio(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=lambda p: -p[0] / (p[0] + p[1]))
    return int("".join(seg(x, y) for x, y in pairs), 2) % MOD

def wrong_length(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=lambda p: -(p[0] + p[1]))
    return int("".join(seg(x, y) for x, y in pairs), 2) % MOD

a, b = [3, 1], [3, 0]           # "111000" và "1"
print("tỉ lệ:", wrong_ratio(a, b), "| độ dài:", wrong_length(a, b), "| đúng:", Solution().maxValue(a, b))
a, b = [1, 3], [1, 4]           # "10" (tỉ lệ 1/2) và "1110000" (tỉ lệ 3/7)
print("tỉ lệ:", wrong_ratio(a, b), "| đúng:", Solution().maxValue(a, b), "-> tỉ lệ cao hơn chưa chắc phải đứng trước")

tỉ lệ: 120 | độ dài: 113 | đúng: 120
tỉ lệ: 368 | đúng: 450 -> tỉ lệ cao hơn chưa chắc phải đứng trước


In [19]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3897)
for _ in range(400):
    n = rng.randint(1, 6)
    a, b = [], []
    for _ in range(n):
        x, y = rng.randint(0, 4), rng.randint(0, 4)
        if x + y == 0: x = 1
        a.append(x); b.append(y)
    r = run_all(a, b)
    assert len(set(r)) == 1, (a, b, r)
print("400 test ngẫu nhiên (có vét cạn hoán vị): OK")

400 test ngẫu nhiên (có vét cạn hoán vị): OK


In [20]:
# 12.5 test lớn đúng ràng buộc: tổng 2*10^5
rng = random.Random(5)
n = 10**5
a = [rng.randint(0, 2) for _ in range(n)]
b = [rng.randint(0, 2) for _ in range(n)]
for i in range(n):
    if a[i] + b[i] == 0: a[i] = 1
t0 = time.time(); x1 = Solution().maxValue(a, b); t1 = time.time()
x2 = key_sort_horner(a, b); x3 = concat_compare(a, b)
print("khớp 3 cách:", x1 == x2 == x3, "| tổng độ dài:", sum(a) + sum(b), "| thời gian Cách 3 < 2s:", t1 - t0 < 2)

khớp 3 cách: True | tổng độ dài: 210857 | thời gian Cách 3 < 2s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cùng số 1 thì sắp số 0 **giảm dần**

In [21]:
def wrong_y_desc(nums1, nums0):
    def key(p):
        x, y = p
        if y == 0: return (0, -x, 0)
        if x > 0: return (1, -x, -y)          # SAI: y phải tăng dần
        return (2, 0, 0)
    return int("".join(seg(x, y) for x, y in sorted(zip(nums1, nums0), key=key)), 2) % MOD

print("sai:", wrong_y_desc([1, 1], [1, 3]), "| đúng:", Solution().maxValue([1, 1], [1, 3]))

sai: 34 | đúng: 40


**Vì sao sai:** `"10" + "1000" = 101000` > `"1000" + "10" = 100010`. Ít số 0 hơn phải đứng trước.

### Lỗi 2 — Đặt đoạn toàn 1 chung nhóm với đoạn hỗn hợp theo `x` giảm dần

Đoạn `"1"` (toàn 1, `x = 1`) bị xếp **sau** đoạn `"110"` (`x = 2`), nhưng `"1" + "110" = 1110` > `"110" + "1" = 1101`.

In [22]:
def wrong_merge_groups(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=lambda p: (-p[0], p[1]))   # SAI: không tách nhóm toàn 1
    return int("".join(seg(x, y) for x, y in pairs), 2) % MOD

print("sai:", wrong_merge_groups([2, 1], [1, 0]), "| đúng:", Solution().maxValue([2, 1], [1, 0]))

sai: 13 | đúng: 14


### Lỗi 3 — Tính giá trị bằng `int(s, 2)` rồi mới `% MOD`… với xâu khổng lồ trong ngôn ngữ khác

Trong Python `int(s, 2)` với `2·10^5` bit vẫn chạy được, nhưng tốn bộ nhớ và **không mang sang** C++/Java. Thói quen tốt: tích lũy theo modulo (`pw` hoặc Horner). Ngoài ra, quên `% MOD` ở cuối sẽ trả về số khổng lồ.

In [23]:
def wrong_no_mod(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=seg_key)
    return int("".join(seg(x, y) for x, y in pairs), 2)            # SAI: thiếu % MOD

v = wrong_no_mod([40], [0])
print("thiếu mod:", v, "| đúng:", Solution().maxValue([40], [0]))

thiếu mod: 1099511627775 | đúng: 511620082


### Lỗi 4 — Trừ con trỏ vị trí sai (quên trừ `y`)

In [24]:
def wrong_pos(nums1, nums0):
    pairs = sorted(zip(nums1, nums0), key=seg_key)
    L = sum(nums1) + sum(nums0); pos = L - 1; ans = 0
    for x, y in pairs:
        for _ in range(x):
            ans += 2 ** pos; pos -= 1
        # SAI: quên pos -= y
    return ans % MOD

# Với [3,1],[0,3] lỗi bị "che" vì đoạn có số 0 đứng cuối; dữ liệu dưới đây làm lộ lỗi:
print("sai:", wrong_pos([1, 1], [2, 1]), "| đúng:", Solution().maxValue([1, 1], [2, 1]))

sai: 24 | đúng: 20


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Sắp xếp lại các mảnh rồi nối để được số/xâu lớn nhất"** ⇒ so sánh `A+B` với `B+A` (LC 179 *Largest Number*, LC 1585…).
- Khi các mảnh có **cấu trúc đặc biệt** (ở đây `1^x 0^y`), rút gọn so sánh thành **khóa tuple** để sắp xếp nhanh và tránh dựng xâu.
- **Tổng độ dài cố định** ⇒ "giá trị lớn nhất" = "từ điển lớn nhất".
- **Số rất lớn theo modulo** ⇒ tiền tính lũy thừa hoặc Horner.

## 15. Biến thể của bài

1. **Giá trị nhỏ nhất**: đảo mọi so sánh — toàn 0 trước, hỗn hợp `x` tăng / `y` giảm…, toàn 1 cuối (kiểm chứng bằng vét cạn).
2. **Đoạn dạng `0^y 1^x`** (0 trước 1): cần lập lại bảng so sánh.
3. **Mảnh là xâu nhị phân tùy ý**: quay lại `cmp_to_key` với `A+B` vs `B+A` (vẫn bắc cầu với xâu).
4. **Không được đổi chỗ, chỉ được đảo (reverse) từng đoạn**: mỗi đoạn chọn `1^x0^y` hoặc `0^y1^x` theo tham lam từ trái.

In [25]:
# Biến thể 1: giá trị nhỏ nhất — dùng so sánh A+B < B+A và kiểm chứng vét cạn
def min_value(nums1, nums0):
    segs = [seg(x, y) for x, y in zip(nums1, nums0)]
    segs.sort(key=cmp_to_key(lambda a, b: -1 if a + b < b + a else (1 if a + b > b + a else 0)))
    return int("".join(segs), 2) % MOD

def brute_min(nums1, nums0):
    segs = [seg(x, y) for x, y in zip(nums1, nums0)]
    return min(int("".join(p), 2) for p in permutations(segs)) % MOD

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 6)
    a = [rng.randint(0, 3) for _ in range(n)]; b = [rng.randint(0, 3) for _ in range(n)]
    a = [x if x + y else 1 for x, y in zip(a, b)]
    assert min_value(a, b) == brute_min(a, b)
print("Biến thể 1 (nhỏ nhất) khớp vét cạn.")

Biến thể 1 (nhỏ nhất) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | Thử mọi thứ tự | `O(n! L)` | `O(L)` | Kiểm thử |
| 2. `cmp_to_key` | `A+B` vs `B+A` | `O(n log n · |đoạn|)` | `O(L)` | Mảnh tùy ý, khuôn tổng quát |
| 3. Khóa + `pw` | Nhóm + `(-x, y)`; cộng `2^pos` | `O(n log n + L)` | `O(n + L)` | **Lời giải chuẩn** |
| 4. Khóa + Horner | Gộp từng đoạn bằng `pow` | `O(n log n + n log L)` | `O(n)` | Khi `L` rất lớn so với `n` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums1 = [1, 1], nums0 = [0, 1]` → ?

**Bài 2 (tính tay).** Sắp xếp các đoạn `"1100"`, `"1"`, `"0"`, `"110"`, `"11"` theo khóa và viết xâu kết quả.

**Bài 3 (điền TODO).** Viết `seg_key_min(p)` cho biến thể **giá trị nhỏ nhất**.

**Bài 4 (tự viết lại).** Viết lại lời giải chính chỉ dùng Horner với một biến `val` và **không** gọi `pow` (gợi ý: nhân dần `2` từng bit — vẫn `O(L)`).

**Bài 5 (mini challenge).** Mỗi đoạn là `0^y 1^x` (0 trước). Tìm khóa sắp xếp cho giá trị **lớn nhất** và kiểm chứng.

In [26]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def seg_key_min(p):
    # TODO: toàn 0 trước, rồi hỗn hợp (gợi ý: x tăng, y giảm), toàn 1 cuối
    pass

In [27]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_value_zero_first(nums1, nums0):
    # TODO: đoạn 0^y 1^x; thử suy ra khóa, đối chiếu cmp_to_key
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đoạn `"1"` (toàn 1) và `"10"`. (2) Toàn 1 trước: `"1" + "10" = "110"`. (3) Đáp án `6`.

**Bài 2.** (1) Toàn 1: `"11"`, `"1"` (x giảm). (2) Hỗn hợp: `"110"` (x=2,y=1), `"1100"` (x=2,y=2). (3) Toàn 0: `"0"` ⇒ `"11" "1" "110" "1100" "0"`.

**Bài 3.** (1) Muốn số 0 càng sớm càng tốt. (2) Đoạn hỗn hợp `1^x0^y`: ít 1 trước (`x` tăng); cùng `x` thì nhiều 0 trước (`y` giảm). (3) Kiểm chứng bằng `brute_min`.

**Bài 4.** (1) Với mỗi bit: `val = val*2 + bit`. (2) Duyệt `x` bit 1 rồi `y` bit 0. (3) `% MOD` mỗi bước.

**Bài 5.** (1) Toàn 1 trước, toàn 0 cuối vẫn đúng. (2) Đoạn hỗn hợp `0^y1^x`: bắt đầu bằng 0 nên ít 0 trước (`y` tăng); cùng `y` thì nhiều 1 trước (`x` giảm). (3) Đối chiếu `cmp_to_key`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
assert Solution().maxValue([1, 1], [0, 1]) == 6
order = sorted([(2, 2), (1, 0), (0, 1), (2, 1), (2, 0)], key=seg_key)
assert "".join(seg(x, y) for x, y in order) == "11" + "1" + "110" + "1100" + "0"
print("Bài 1, 2 OK:", [seg(x, y) for x, y in order])

Bài 1, 2 OK: ['11', '1', '110', '1100', '0']


In [29]:
# Bài 3
def seg_key_min_sol(p):
    x, y = p
    if x == 0: return (0, -y, 0)
    if y > 0: return (1, x, -y)
    return (2, 0, 0)

def min_by_key(a, b):
    return int("".join(seg(x, y) for x, y in sorted(zip(a, b), key=seg_key_min_sol)), 2) % MOD

rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 6)
    a = [rng.randint(0, 3) for _ in range(n)]; b = [rng.randint(0, 3) for _ in range(n)]
    a = [x if x + y else 1 for x, y in zip(a, b)]
    assert min_by_key(a, b) == brute_min(a, b)
print("Bài 3 OK")

Bài 3 OK


In [30]:
# Bài 4
def sol_horner_bits(nums1, nums0):
    val = 0
    for x, y in sorted(zip(nums1, nums0), key=seg_key):
        for _ in range(x): val = (val * 2 + 1) % MOD
        for _ in range(y): val = val * 2 % MOD
    return val

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 30)
    a = [rng.randint(0, 50) for _ in range(n)]; b = [rng.randint(0, 50) for _ in range(n)]
    a = [x if x + y else 1 for x, y in zip(a, b)]
    assert sol_horner_bits(a, b) == Solution().maxValue(a, b)
print("Bài 4 OK")

Bài 4 OK


In [31]:
# Bài 5
def seg0(x, y): return '0' * y + '1' * x
def key_zero_first(p):
    x, y = p
    if y == 0: return (0, -x, 0)          # toàn 1
    if x > 0: return (1, y, -x)           # hỗn hợp 0^y 1^x: ít 0 trước, nhiều 1 trước
    return (2, 0, 0)                      # toàn 0

def max_zero_first(a, b):
    return int("".join(seg0(x, y) for x, y in sorted(zip(a, b), key=key_zero_first)), 2) % MOD

def brute_zero_first(a, b):
    return max(int("".join(p), 2) for p in permutations([seg0(x, y) for x, y in zip(a, b)])) % MOD

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 6)
    a = [rng.randint(0, 3) for _ in range(n)]; b = [rng.randint(0, 3) for _ in range(n)]
    a = [x if x + y else 1 for x, y in zip(a, b)]
    assert max_zero_first(a, b) == brute_zero_first(a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def key(p):
    x, y = p
    if y == 0: return (0, -x, 0)   # toàn 1
    if x > 0:  return (1, -x, y)   # hỗn hợp
    return (2, 0, 0)               # toàn 0
pairs = sorted(zip(nums1, nums0), key=key)
# cộng 2^pos cho từng bit 1, pos chạy từ L-1 xuống; nhớ pos -= y
```

**Checklist:**

- [ ] Toàn 1 đứng **trước** mọi đoạn hỗn hợp (kể cả đoạn hỗn hợp có nhiều 1 hơn)?
- [ ] Hỗn hợp: `x` **giảm**, cùng `x` thì `y` **tăng**?
- [ ] Toàn 0 ở **cuối**?
- [ ] Con trỏ vị trí trừ cả số chữ 0 của mỗi đoạn?
- [ ] Lấy `% MOD`?